In [1]:
import pandas as pd

In [3]:
df = pd.read_csv('D:/Project/AI That Explains Medical Images Like a Doctor/Another way/Dataset/stage_2_train_labels.csv')

In [4]:
df2 = pd.read_csv('D:\Project\AI That Explains Medical Images Like a Doctor\Another way\Dataset\stage_2_detailed_class_info.csv')

In [5]:
df.head()

,patientId,x,y,width,height,Target
0,0004cfab-14fd-4e49-80ba-63a80b6bddd6,NaN,NaN,NaN,NaN,0
1,00313ee0-9eaa-42f4-b0ab-c148ed3241cd,NaN,NaN,NaN,NaN,0
2,00322d4d-1c29-4943-afc9-b6754be640eb,NaN,NaN,NaN,NaN,0
3,003d8fa0-6bf1-40ed-b54c-ac657f8495c5,NaN,NaN,NaN,NaN,0
4,00436515-870c-4b36-a041-de91049b9ab4,264.0,152.0,213.0,379.0,1


In [24]:
df['patientId'][0]

'0004cfab-14fd-4e49-80ba-63a80b6bddd6'

In [6]:
df2.head()

,patientId,class
0,0004cfab-14fd-4e49-80ba-63a80b6bddd6,No Lung Opacity / Not Normal
1,00313ee0-9eaa-42f4-b0ab-c148ed3241cd,No Lung Opacity / Not Normal
2,00322d4d-1c29-4943-afc9-b6754be640eb,No Lung Opacity / Not Normal
3,003d8fa0-6bf1-40ed-b54c-ac657f8495c5,Normal
4,00436515-870c-4b36-a041-de91049b9ab4,Lung Opacity


In [25]:
df2[df2['class'] == 'No Lung Opacity / Not Normal'] = 'No Lung Opacity'

In [21]:
print(df.shape)
print(df2.shape)

(30227, 6)
(30227, 2)


In [18]:
df2['class'].unique()

array(['No Lung Opacity / Not Normal', 'Normal', 'Lung Opacity'],
      dtype=object)

In [14]:
df['Target'].value_counts()

Target
0    20672
1     9555
Name: count, dtype: int64

In [16]:
df2[(df2['patientId'] == df['patientId']) & df['Target'] == 1]['class'].value_counts()

class
Lung Opacity    9555
Name: count, dtype: int64

In [7]:
df2['class'].value_counts()

class
No Lung Opacity / Not Normal    11821
Lung Opacity                     9555
Normal                           8851
Name: count, dtype: int64

In [28]:
import os
import pandas as pd
import pydicom
import numpy as np
import cv2

# df and df2 already loaded
# df['patient_id']
# df2['class']

dicom_dir = r"D:\Project\AI That Explains Medical Images Like a Doctor\Another way\Dataset\stage_2_train_images"
output_root = r"D:\Project\AI That Explains Medical Images Like a Doctor\Another way\Dataset\images"

os.makedirs(output_root, exist_ok=True)

i=0
for idx in range(len(df)):
    patient_id = str(df.loc[idx, 'patientId'])
    class_name = str(df2.loc[idx, 'class'])

    print(f"Processing {i+1}/{len(df)}: {patient_id} -> {class_name}")
    i += 1
    dicom_path = os.path.join(dicom_dir, patient_id + ".dcm")

    # Skip if DICOM not found
    if not os.path.exists(dicom_path):
        print(f"❌ Missing: {patient_id}.dcm")
        continue

    # Create class folder
    class_folder = os.path.join(output_root, class_name)
    os.makedirs(class_folder, exist_ok=True)

    # Read DICOM
    ds = pydicom.dcmread(dicom_path)
    image = ds.pixel_array.astype(np.float32)

    # Normalize
    image = (image - image.min()) / (image.max() - image.min())
    image = (image * 255).astype(np.uint8)

    # Save image
    output_path = os.path.join(class_folder, patient_id + ".png")
    cv2.imwrite(output_path, image)

print("✅ All matching DICOM files processed and saved")


Processing 1/30227: 0004cfab-14fd-4e49-80ba-63a80b6bddd6 -> No Lung Opacity
Processing 2/30227: 00313ee0-9eaa-42f4-b0ab-c148ed3241cd -> No Lung Opacity
Processing 3/30227: 00322d4d-1c29-4943-afc9-b6754be640eb -> No Lung Opacity
Processing 4/30227: 003d8fa0-6bf1-40ed-b54c-ac657f8495c5 -> Normal
Processing 5/30227: 00436515-870c-4b36-a041-de91049b9ab4 -> Lung Opacity
Processing 6/30227: 00436515-870c-4b36-a041-de91049b9ab4 -> Lung Opacity
Processing 7/30227: 00569f44-917d-4c86-a842-81832af98c30 -> No Lung Opacity
Processing 8/30227: 006cec2e-6ce2-4549-bffa-eadfcd1e9970 -> No Lung Opacity
Processing 9/30227: 00704310-78a8-4b38-8475-49f4573b2dbb -> Lung Opacity
Processing 10/30227: 00704310-78a8-4b38-8475-49f4573b2dbb -> Lung Opacity
Processing 11/30227: 008c19e8-a820-403a-930a-bc74a4053664 -> No Lung Opacity
Processing 12/30227: 009482dc-3db5-48d4-8580-5c89c4f01334 -> Normal
Processing 13/30227: 009eb222-eabc-4150-8121-d5a6d06b8ebf -> Normal
Processing 14/30227: 00a85be6-6eb0-421d-8acf-ff

KeyboardInterrupt: 

In [29]:
import os
import pandas as pd
import pydicom
import numpy as np
import cupy as cp
import cv2

# 🔥 Force GPU usage (will crash if GPU not detected)
assert cp.cuda.runtime.getDeviceCount() > 0, "❌ GPU NOT DETECTED"

dicom_dir = r"D:\Project\AI That Explains Medical Images Like a Doctor\Another way\Dataset\stage_2_train_images"
output_root = r"D:\Project\AI That Explains Medical Images Like a Doctor\Another way\Dataset\images"

os.makedirs(output_root, exist_ok=True)

i = 0
for idx in range(len(df)):
    patient_id = str(df.loc[idx, 'patientId'])
    class_name = str(df2.loc[idx, 'class'])

    print(f"Processing {i+1}/{len(df)}: {patient_id} -> {class_name}")
    i += 1

    dicom_path = os.path.join(dicom_dir, patient_id + ".dcm")

    if not os.path.exists(dicom_path):
        print(f"❌ Missing: {patient_id}.dcm")
        continue

    class_folder = os.path.join(output_root, class_name)
    os.makedirs(class_folder, exist_ok=True)

    # ❌ CPU (cannot be changed)
    ds = pydicom.dcmread(dicom_path)
    image = ds.pixel_array.astype(np.float32)

    # ✅ CPU → GPU
    gpu_img = cp.asarray(image)

    # 🚀 GPU NORMALIZATION
    gpu_img = (gpu_img - cp.min(gpu_img)) / (cp.max(gpu_img) - cp.min(gpu_img))
    gpu_img = (gpu_img * 255).astype(cp.uint8)

    # ✅ GPU → CPU (only for saving)
    final_img = cp.asnumpy(gpu_img)

    output_path = os.path.join(class_folder, patient_id + ".png")
    cv2.imwrite(output_path, final_img)

print("🔥 All matching DICOM files processed using GPU")


Processing 1/30227: 0004cfab-14fd-4e49-80ba-63a80b6bddd6 -> No Lung Opacity
Processing 2/30227: 00313ee0-9eaa-42f4-b0ab-c148ed3241cd -> No Lung Opacity
Processing 3/30227: 00322d4d-1c29-4943-afc9-b6754be640eb -> No Lung Opacity
Processing 4/30227: 003d8fa0-6bf1-40ed-b54c-ac657f8495c5 -> Normal
Processing 5/30227: 00436515-870c-4b36-a041-de91049b9ab4 -> Lung Opacity
Processing 6/30227: 00436515-870c-4b36-a041-de91049b9ab4 -> Lung Opacity
Processing 7/30227: 00569f44-917d-4c86-a842-81832af98c30 -> No Lung Opacity
Processing 8/30227: 006cec2e-6ce2-4549-bffa-eadfcd1e9970 -> No Lung Opacity
Processing 9/30227: 00704310-78a8-4b38-8475-49f4573b2dbb -> Lung Opacity
Processing 10/30227: 00704310-78a8-4b38-8475-49f4573b2dbb -> Lung Opacity
Processing 11/30227: 008c19e8-a820-403a-930a-bc74a4053664 -> No Lung Opacity
Processing 12/30227: 009482dc-3db5-48d4-8580-5c89c4f01334 -> Normal
Processing 13/30227: 009eb222-eabc-4150-8121-d5a6d06b8ebf -> Normal
Processing 14/30227: 00a85be6-6eb0-421d-8acf-ff

In [1]:
import os
import pydicom
import numpy as np
import cv2

input_dir = "D:\Project\AI That Explains Medical Images Like a Doctor\Another way\Dataset\stage_2_test_images"
output_dir = "images/test"
os.makedirs(output_dir, exist_ok=True)
i=0
for file in os.listdir(input_dir):
    if file.endswith(".dcm"):
        i+=1
        print(f"Processing file {i}: {file}")     
        path = os.path.join(input_dir, file)
        ds = pydicom.dcmread(path)

        image = ds.pixel_array.astype(np.float32)
        image = (image - image.min()) / (image.max() - image.min())
        image = (image * 255).astype(np.uint8)

        out_path = os.path.join(output_dir, file.replace(".dcm", ".png"))
        cv2.imwrite(out_path, image)

print("All DICOM files converted!")


Processing file 1: 0000a175-0e68-4ca4-b1af-167204a7e0bc.dcm
Processing file 2: 0005d3cc-3c3f-40b9-93c3-46231c3eb813.dcm
Processing file 3: 000686d7-f4fc-448d-97a0-44fa9c5d3aa6.dcm
Processing file 4: 000e3a7d-c0ca-4349-bb26-5af2d8993c3d.dcm
Processing file 5: 00100a24-854d-423d-a092-edcf6179e061.dcm
Processing file 6: 0015597f-2d69-4bc7-b642-5b5e01534676.dcm
Processing file 7: 001b0c51-c7b3-45c1-9c17-fa7594cab96e.dcm
Processing file 8: 0022bb50-bf6c-4185-843e-403a9cc1ea80.dcm
Processing file 9: 00271e8e-aea8-4f0a-8a34-3025831f1079.dcm
Processing file 10: 0028450f-5b8e-4695-9416-8340b6f686b0.dcm
Processing file 11: 002bcde0-d8da-4931-ab04-5d724e30261b.dcm
Processing file 12: 002fcb77-ef76-4626-ab34-5070f15c20db.dcm
Processing file 13: 003206b4-bd4a-4684-8d49-76f4cb713a30.dcm
Processing file 14: 00330f7f-d114-4eb2-9c6e-558eeb3084a1.dcm
Processing file 15: 00342ae8-ff81-4229-adf6-6a2ab711707b.dcm
Processing file 16: 003d17f0-bd8a-485c-bc8b-daec33f53efa.dcm
Processing file 17: 003dba79-1b1d